In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Import libraries

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torch.nn.functional as F
from torch.utils.data import random_split
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
%matplotlib inline

## Reading files

In [ ]:
train_ds=pd.read_csv("../input/fashionmnist/fashion-mnist_train.csv")
test_ds=pd.read_csv("../input/fashionmnist/fashion-mnist_test.csv")

In [ ]:
test_ds.head()

## Define dependent and independent features for test data

In [ ]:
y=test_ds['label'].to_numpy()
x = test_ds.drop('label',axis=1).values

## converting  test data to tensor

In [ ]:
x=torch.tensor(x)
y=torch.tensor(y)

In [ ]:
train_ds.head()

In [ ]:
train_ds.shape

In [ ]:
len(test_ds)

## define dependent and independent features


In [ ]:
label=train_ds['label']   #pandas series

In [ ]:
train_ds=train_ds.drop('label',axis=1)

In [ ]:
plt.imshow(train_ds[0:1].values.reshape(28,28))
plt.axis("off")
print(label[0])

## converting to numpy  arrays

In [ ]:
train_ds=train_ds.values
label=label.to_numpy()


In [ ]:
np.max(train_ds[0])

In [ ]:
label=torch.tensor(label)
train_ds=torch.tensor(train_ds)

In [ ]:
traiin_ds=TensorDataset(train_ds,label)

In [ ]:
test_ds=TensorDataset(x,y)

In [ ]:
traiin_ds[0:2]

In [ ]:
train_ds,val_ds=random_split(traiin_ds, (50000,10000))

In [ ]:
img,lab = train_ds[7]
print(img.shape)
print(lab)
plt.imshow(img.reshape(28,28))

## create batches

In [ ]:
batch_size=128

In [ ]:
train_loader=DataLoader(train_ds,batch_size,shuffle=True)
val_loader=  DataLoader(train_ds,batch_size*2)

In [ ]:
test_loader=DataLoader(test_ds,batch_size*2)

## Model

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits
    
model=NeuralNetwork().to('cuda')

In [ ]:
model.parameters()

In [ ]:
optimizer=torch.optim.SGD
opt= optimizer(model.parameters(),lr=0.001)
loss_func=F.cross_entropy

## model training

In [ ]:
def fit(epochs,model,data):
    
    hist=[]
    for epoch in range(epochs):
        for img,label in data:
#             print(img.shape)
            img,label=img.to("cuda"),label.to('cuda')
            out = model(img/255)
            loss= loss_func(out,label)
            loss.backward()
            opt.step()
            opt.zero_grad()
            hist.append(loss)
            
        
        if (epoch+1)%10==0:
            print(f"epoch : [{epoch+1}/{epochs}] ; loss : {loss.item()}")
    

In [ ]:
fit(100,model,train_loader)

In [ ]:
def acc(data):
    accy=[]
    for img ,label in data:
        img,label = img.to('cuda'),label.to('cuda')
        out=model(img/255)
#         out=F.softmax(out)
        _,pred_index=torch.max(out,dim=1)
        x=torch.sum(pred_index==label)/len(pred_index)
        x=x*100
        x=x.to("cpu").numpy()
        accy.append(x)
        
        
    return np.mean(accy)  
        

In [ ]:
acc(val_loader)

In [ ]:
acc(test_loader)
    

## Do Upvote